In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

In [ ]:
train_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')
train_data = train_data.drop(columns=['Id'])
test_id = test_data['Id']
test_data = test_data.drop(columns=['Id'])
def PlotMissing(df):
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    plt.figure(figsize=(12, 6))
    sns.barplot(x=missing.index, y=missing.values)
    plt.xticks(rotation=90)
    plt.title("Missing Values per Column")
    plt.tight_layout()
    plt.show()
PlotMissing(train_data)
PlotMissing(test_data)
print("lets drop those who have more missing data then 1300")
threshold = 1300
drop = train_data.columns[train_data.isnull().sum() > threshold]
print(f"Dropping columns: {list(drop)}")
train_data = train_data.drop(columns=drop)
test_data = test_data.drop(columns=drop)
print("lest fill them up with mode and mean , medaian ")
def fillValues(df):
    df = df.copy()
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns
    cat_cols = df.select_dtypes(include=['object']).columns
    num_imputer = SimpleImputer(strategy='median')
    df[num_cols] = num_imputer.fit_transform(df[num_cols])
    cat_imputer = SimpleImputer(strategy='constant', fill_value='None')
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])
    return df
train_data = fillValues(train_data)
test_data = fillValues(test_data)
# PlotMissing(train_data)
# PlotMissing(test_data)
cat_cols = list(train_data.dtypes[train_data.dtypes == 'object'].index)
train_data = pd.get_dummies(train_data,columns=cat_cols,drop_first=True)
test_data = pd.get_dummies(test_data,columns=cat_cols,drop_first=True)
print("good no missing data and all numerical now lest train model")

X = train_data.drop(columns=['SalePrice'])
y = train_data['SalePrice']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
test_data = test_data.reindex(columns=X_train.columns, fill_value=0)
n_estimators = 600
models = [
    RandomForestRegressor(n_estimators=n_estimators, random_state=42),
    LGBMRegressor(n_estimators=n_estimators, random_state=42),
    XGBRegressor(n_estimators=n_estimators, random_state=42, verbosity=0),
    CatBoostRegressor(iterations=n_estimators, random_seed=42, verbose=0),
    LinearRegression()
]
for model in models:
    model_name = model.__class__.__name__
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    print(f"{model_name} RMSE: {rmse:.2f}")
model = models[3]
test_pred = model.predict(test_data)
submission = pd.DataFrame({
    "Id":test_id,
    "SalePrice":test_pred
})
submission.to_csv('submission.csv', index=False)